# 07 — XAI: Similarity Maps (Reproduce Figure 2)

Implements the Riniker & Landrum (2013) atom contribution method:
- Systematically removes fingerprint bits corresponding to atoms/fragments
- Measures how each removal shifts the model's predicted probability
- Normalizes and visualizes as similarity maps

Test molecules (both excluded from training sets per paper):
- **Doxorubicin** (cytotoxic) → should be mostly **red**
- **Ibuprofen** (non-cytotoxic) → should be mostly **green**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, DataStructs
from rdkit.Chem.Draw import SimilarityMaps
from joblib import load
from IPython.display import display, Image
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data/results/xai', exist_ok=True)

In [ ]:
TEST_MOLECULES = {
    'Doxorubicin': 'COc1cccc2c1C(=O)c1c(O)c3c(c(O)c1C2=O)C[C@@](O)(C(=O)CO)C[C@H]3O[C@H]1C[C@H](N)[C@H](O)[C@H](C)O1',
    'Ibuprofen':   'CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O',
}

In [ ]:
def get_ecfp4_prob(model, mol, nbits=1024):
    """Predict cytotoxicity probability for an RDKit mol object."""
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=nbits)
    arr = np.zeros((1, nbits), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr[0])
    prob = model.predict_proba(arr)[0, 1]  # P(cytotoxic)
    return prob


def atom_contributions_ecfp4(model, mol, nbits=1024, radius=2):
    """
    Riniker & Landrum (2013): for each atom, compute the change in prediction
    probability when the bits associated with that atom are zeroed out.
    Returns a list of per-atom weights (positive = promotes cytotoxic prediction).
    """
    base_fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nbits)
    base_arr = np.zeros((nbits,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(base_fp, base_arr)
    base_prob = model.predict_proba(base_arr.reshape(1, -1))[0, 1]

    # Map each bit to the atoms that set it
    bit_info = {}
    AllChem.GetMorganFingerprintAsBitVect(mol, radius, nbits, bitInfo=bit_info)

    # For each atom, collect the bits it participates in
    atom_bits = {atom.GetIdx(): set() for atom in mol.GetAtoms()}
    for bit, atom_radius_list in bit_info.items():
        for atom_idx, _ in atom_radius_list:
            atom_bits[atom_idx].add(bit)

    weights = []
    for atom_idx in range(mol.GetNumAtoms()):
        bits = atom_bits.get(atom_idx, set())
        if not bits:
            weights.append(0.0)
            continue
        # Zero out all bits for this atom
        perturbed = base_arr.copy()
        for b in bits:
            perturbed[b] = 0
        perturbed_prob = model.predict_proba(perturbed.reshape(1, -1))[0, 1]
        # Positive weight → removing these bits decreases cytotoxic probability
        # → atom contributes TO cytotoxicity (red)
        weights.append(base_prob - perturbed_prob)

    return weights, base_prob

In [ ]:
DATASETS = ['3T3', 'HEK293']

fig, axes = plt.subplots(len(TEST_MOLECULES), len(DATASETS), figsize=(5 * len(DATASETS), 4 * len(TEST_MOLECULES)))
if len(TEST_MOLECULES) == 1:
    axes = [axes]

for col_idx, cell_line in enumerate(DATASETS):
    model_path = f'../models/{cell_line}_lgbm_ecfp4.joblib'
    if not os.path.exists(model_path):
        print(f'Model not found for {cell_line} — run notebook 04 first.')
        continue
    model = load(model_path)
    print(f'\n=== {cell_line} ===')

    for row_idx, (mol_name, smi) in enumerate(TEST_MOLECULES.items()):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            print(f'  Could not parse SMILES for {mol_name}')
            continue

        weights, prob = atom_contributions_ecfp4(model, mol)
        pred_label = 'Cytotoxic' if prob >= 0.5 else 'Non-cytotoxic'
        print(f'  {mol_name}: P(cytotoxic)={prob:.3f} → {pred_label}')

        # Normalize weights to [-1, 1]
        max_w = max(abs(w) for w in weights) if any(weights) else 1.0
        norm_weights = [w / max_w if max_w > 0 else 0.0 for w in weights]

        # Draw similarity map
        ax = axes[row_idx][col_idx]
        fig_tmp, ax_tmp = plt.subplots(figsize=(4, 3))
        SimilarityMaps.GetSimilarityMapFromWeights(
            mol, norm_weights,
            colorMap='RdYlGn_r',   # red=cytotoxic, green=non-cytotoxic
            alpha=0.5,
            ax=ax_tmp,
        )
        ax_tmp.set_title(f'{mol_name}\n{cell_line} | P={prob:.2f} ({pred_label})', fontsize=9)
        plt.tight_layout()

        out_path = f'../data/results/xai/{mol_name}_{cell_line}.png'
        fig_tmp.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig_tmp)
        print(f'  Saved: {out_path}')

plt.close(fig)
print('\nAll XAI maps saved to ../data/results/xai/')

In [ ]:
# Display all saved maps inline
from IPython.display import Image, display
import glob

for f in sorted(glob.glob('../data/results/xai/*.png')):
    print(f)
    display(Image(filename=f, width=400))